### Loading Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Loading Datasets

In [2]:
inventory = pd.read_csv("bokku dataset/inventory.csv")
products = pd.read_csv("bokku dataset/products.csv")
sales = pd.read_csv("bokku dataset/sales_transactions.csv")
stores = pd.read_csv("bokku dataset/stores.csv")
suppliers = pd.read_csv("bokku dataset/suppliers.csv")
warehouses = pd.read_csv("bokku dataset/warehouses.csv")

In [3]:
sales.head(2)

,transaction_id,store_id,product_id,quantity_sold,sales_amount,transaction_date
0,T00001,ST05970,P02643,12,115840.56,2025-04-27
1,T00002,ST04467,P03411,1,9896.23,2025-05-18


In [4]:
products.head(2)

,product_id,product_name,category,brand,unit_price_ngn,supplier_id
0,P00001,Peak Product 1,Food,Nestle,6273.32,S02287
1,P00002,Peak Product 2,Cleaning,Peak,14844.21,S00521


### Inventory Turnover Analysis

In [5]:
# formula for Inventory Turnover Analysis - COGS / Average inventory value

cogs = sales['sales_amount'].sum()
cogs

np.float64(3212540071.2)

In [6]:
# To calculate inventory value is Unit_price x current_stock and since the dataset has unit_price_ngn in products table
# and current stock in inventory table, so i merge only unit_price_ngn to inventory table based on product_id 
# then multiply current_stock by unit_price_ngn

In [7]:
merge_inventory = inventory.merge(
    products[['product_id', 'unit_price_ngn']],
    on='product_id',
    how='left'
)

In [8]:
merge_inventory.head()

,inventory_id,product_id,warehouse_id,current_stock,reorder_level,max_capacity,last_updated,unit_price_ngn
0,I00001,P03994,W04507,1005,276,4027,2026-03-14,20048.75
1,I00002,P01759,W09868,602,221,9745,2026-04-29,15818.61
2,I00003,P07241,W09951,1484,499,1095,2026-05-27,20576.15
3,I00004,P00229,W09085,3559,306,8374,2026-04-03,14742.30
4,I00005,P02681,W06466,2061,113,5267,2026-05-07,14104.63


In [9]:
merge_inventory['inventory_value'] = merge_inventory['current_stock'] * merge_inventory['unit_price_ngn']

In [10]:
average_merge_inventory = merge_inventory['inventory_value'].mean()

In [11]:
average_merge_inventory

np.float64(31836065.561720997)

In [12]:
turnover_inventory = cogs / average_merge_inventory

turnover_inventory


np.float64(100.90882822727596)

##### the result of this Turnover Inventory is too high, because we use unit_price_ngn which is the selling price instead of using the cost price, which is not available in the dataset

### Identify Fast Movers and Identify Slow Movers

In [128]:
#first we groupby quantity_sold on product_id, then rank the products, rank the product, classify them

product_sales = sales.groupby('product_id')['quantity_sold'].sum().reset_index()

In [130]:
product_sales.head(2)

,product_id,quantity_sold
0,P00001,22
1,P00003,44


In [132]:
product_sales['rank'] = product_sales['quantity_sold'].rank(method='dense', ascending=False)

In [133]:
product_sales.head(2)

,product_id,quantity_sold,rank
0,P00001,22,140
1,P00003,44,118


In [134]:
product_sales['movement'] = pd.qcut(
    product_sales['quantity_sold'],
    q=[0, 0.2, 0.8, 1],
    labels=['Slow Mover', 'Medium Mover', 'Fast Mover']
)

In [139]:
product_sales.head(2)

,product_id,quantity_sold,rank,movement
0,P00001,22,140,Medium Mover
1,P00003,44,118,Medium Mover


### ABC Analysis - Business Question => Which products contribute most revenue?

In [13]:
# To calculate ABC analysis, we find the product revenue, sort them, Cumulative %, Categorize, and Count Products

product_revenue = sales.groupby('product_id')['sales_amount'].sum().reset_index()

In [14]:
product_revenue = product_revenue.sort_values("sales_amount", ascending=False)

In [15]:
product_revenue['RevenuePct'] = product_revenue['sales_amount'] / product_revenue['sales_amount'].sum()

In [16]:
product_revenue['CumPct'] = product_revenue['RevenuePct'].cumsum()

In [17]:
product_revenue.head(2)

,product_id,sales_amount,RevenuePct,CumPct
2755,P04363,3034757.65,0.000945,0.000945
3665,P05849,2949148.28,0.000918,0.001863


In [18]:
# creating to classify

def classify(x):
    if x <= 0.80:
        return "A"
    elif x <= 0.95:
        return "B"
    else:
        return "C"

In [19]:
product_revenue['ABC'] = product_revenue['CumPct'].apply(classify)

In [20]:
product_revenue.head(2)

,product_id,sales_amount,RevenuePct,CumPct,ABC
2755,P04363,3034757.65,0.000945,0.000945,A
3665,P05849,2949148.28,0.000918,0.001863,A


In [21]:
#counting the number in each category

product_revenue['ABC'].value_counts()

ABC
A    2905
C    1824
B    1552
Name: count, dtype: int64

### Reorder Point Analysis - Business Question => Which product require replenishment?

In [51]:
daily_demand = sales.groupby("product_id")['quantity_sold'].sum().reset_index()

In [52]:
daily_demand.head(2)

,product_id,quantity_sold
0,P00001,22
1,P00003,44


In [53]:
daily_demand['avg_daily_demand'] = daily_demand['quantity_sold'] / 365

In [54]:
daily_demand.head(2)

,product_id,quantity_sold,avg_daily_demand
0,P00001,22,0.060274
1,P00003,44,0.120548


In [29]:
# i need to lead time column which is in suppliers table, to connect quantity_sold to lead time table, to be able to calculaate for safety stock
# then i will merge the products (product_id) to suppliers table on supplier_id, then merge the supplier table to quantity daily demand 

In [55]:
product_merge = products.merge(suppliers, on='supplier_id', how='left')

In [56]:
product_merge.head(2)

,product_id,product_name,category,brand,unit_price_ngn,supplier_id,supplier_name,lead_time_days,supplier_rating
0,P00001,Peak Product 1,Food,Nestle,6273.32,S02287,Dangote Sugar 2287,11,4.1
1,P00002,Peak Product 2,Cleaning,Peak,14844.21,S00521,Dangote Sugar 521,10,3.5


In [57]:
# Now I will merge product_merge table to daily demand table to calculate for safety stock then calulate for reorder pointa 
# to add quantity_sold to product_merge table from daily_demand

In [58]:
daily_demand_merge_product = pd.merge(product_merge, daily_demand, on="product_id", how="left")

In [59]:
daily_demand_merge_product.head()

,product_id,product_name,category,brand,unit_price_ngn,supplier_id,supplier_name,lead_time_days,supplier_rating,quantity_sold,avg_daily_demand
0,P00001,Peak Product 1,Food,Nestle,6273.32,S02287,Dangote Sugar 2287,11,4.1,22.0,0.060274
1,P00002,Peak Product 2,Cleaning,Peak,14844.21,S00521,Dangote Sugar 521,10,3.5,NaN,NaN
2,P00003,Dangote Product 3,Food,Golden Penny,5969.99,S09864,Nestle Nigeria 9864,13,4.1,44.0,0.120548
3,P00004,Dangote Product 4,Frozen Foods,Golden Penny,17957.29,S08929,Flour Mills Nigeria 8929,2,4.4,NaN,NaN
4,P00005,Milo Product 5,Beverages,Dettol,14813.79,S00107,Flour Mills Nigeria 107,7,4.0,46.0,0.126027


In [ ]:
# Left fix the missing value, Option 1: i can remove the missing value, 
#but i will go will option 2 which taking the median to fillna() for quantity_sold

In [60]:
daily_demand_merge_product['quantity_sold'] = daily_demand_merge_product['quantity_sold'].fillna(daily_demand_merge_product['quantity_sold'].median())

In [61]:
daily_demand_merge_product.head()

,product_id,product_name,category,brand,unit_price_ngn,supplier_id,supplier_name,lead_time_days,supplier_rating,quantity_sold,avg_daily_demand
0,P00001,Peak Product 1,Food,Nestle,6273.32,S02287,Dangote Sugar 2287,11,4.1,22.0,0.060274
1,P00002,Peak Product 2,Cleaning,Peak,14844.21,S00521,Dangote Sugar 521,10,3.5,37.0,NaN
2,P00003,Dangote Product 3,Food,Golden Penny,5969.99,S09864,Nestle Nigeria 9864,13,4.1,44.0,0.120548
3,P00004,Dangote Product 4,Frozen Foods,Golden Penny,17957.29,S08929,Flour Mills Nigeria 8929,2,4.4,37.0,NaN
4,P00005,Milo Product 5,Beverages,Dettol,14813.79,S00107,Flour Mills Nigeria 107,7,4.0,46.0,0.126027


In [62]:
# fixing the missing value for avg_daily_demand

daily_demand_merge_product['avg_daily_demand'] = daily_demand_merge_product['avg_daily_demand'].fillna(daily_demand_merge_product['avg_daily_demand'].median())

In [63]:
daily_demand_merge_product.head(2)

,product_id,product_name,category,brand,unit_price_ngn,supplier_id,supplier_name,lead_time_days,supplier_rating,quantity_sold,avg_daily_demand
0,P00001,Peak Product 1,Food,Nestle,6273.32,S02287,Dangote Sugar 2287,11,4.1,22.0,0.060274
1,P00002,Peak Product 2,Cleaning,Peak,14844.21,S00521,Dangote Sugar 521,10,3.5,37.0,0.101370


In [68]:
#now let calculate for safety stock => average daily daily demand + lead time

daily_demand_merge_product['safety_stock'] = (daily_demand_merge_product['avg_daily_demand'] * daily_demand_merge_product['lead_time_days']).round().astype(int)

In [70]:
daily_demand_merge_product.head(2)

,product_id,product_name,category,brand,unit_price_ngn,supplier_id,supplier_name,lead_time_days,supplier_rating,quantity_sold,avg_daily_demand,safety_stock
0,P00001,Peak Product 1,Food,Nestle,6273.32,S02287,Dangote Sugar 2287,11,4.1,22.0,0.060274,1
1,P00002,Peak Product 2,Cleaning,Peak,14844.21,S00521,Dangote Sugar 521,10,3.5,37.0,0.101370,1


### Reorder Point Analysis - Business Question => Which products require Replenishment?

In [82]:
# formula for reorder point = (Avg_daily_demand x lead_time) + safety_stock


daily_demand_merge_product['reorder_level'] = (daily_demand_merge_product['avg_daily_demand'] * daily_demand_merge_product['lead_time_days']) + daily_demand_merge_product['safety_stock'].round().astype(int)


In [83]:
daily_demand_merge_product.head(2)

,product_id,product_name,category,brand,unit_price_ngn,supplier_id,supplier_name,lead_time_days,supplier_rating,quantity_sold,avg_daily_demand,safety_stock,reorder_level
0,P00001,Peak Product 1,Food,Nestle,6273.32,S02287,Dangote Sugar 2287,11,4.1,22.0,0.060274,1,1.663014
1,P00002,Peak Product 2,Cleaning,Peak,14844.21,S00521,Dangote Sugar 521,10,3.5,37.0,0.101370,1,2.013699


### Demand Forecasting - Business Question => What will demand look like next month?

In [85]:
# we are using sales table

sales.head(2)

,transaction_id,store_id,product_id,quantity_sold,sales_amount,transaction_date
0,T00001,ST05970,P02643,12,115840.56,2025-04-27
1,T00002,ST04467,P03411,1,9896.23,2025-05-18


In [86]:
# extract the year, month, and day from transaction_date, but first let check the data type and convert to datetime format

sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    10000 non-null  object 
 1   store_id          10000 non-null  object 
 2   product_id        10000 non-null  object 
 3   quantity_sold     10000 non-null  int64  
 4   sales_amount      10000 non-null  float64
 5   transaction_date  10000 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 468.9+ KB


In [87]:
sales['transaction_date'] = pd.to_datetime(sales['transaction_date'])

In [88]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    10000 non-null  object        
 1   store_id          10000 non-null  object        
 2   product_id        10000 non-null  object        
 3   quantity_sold     10000 non-null  int64         
 4   sales_amount      10000 non-null  float64       
 5   transaction_date  10000 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(1), object(3)
memory usage: 468.9+ KB


In [89]:
sales['year'] = sales['transaction_date'].dt.year
sales['month'] = sales['transaction_date'].dt.month
sales['day'] = sales['transaction_date'].dt.day

In [90]:
sales.head(2)

,transaction_id,store_id,product_id,quantity_sold,sales_amount,transaction_date,year,month,day
0,T00001,ST05970,P02643,12,115840.56,2025-04-27,2025,4,27
1,T00002,ST04467,P03411,1,9896.23,2025-05-18,2025,5,18


In [108]:
# to calculate the forecast demand, we can simple use moving average method, first, i need to groupby sales per month

pd.set_option('display.float_format', '{:,.0f}'.format)

monthly_sales = sales.groupby(['year', 'month'])['sales_amount'].sum().reset_index()

monthly_sales['sales_amount'] = monthly_sales['sales_amount']


In [109]:
monthly_sales.head(2)

,year,month,sales_amount
0,2025,1,"285,044,244"
1,2025,2,"232,519,409"


In [110]:
#Monthly Sales Forecast

monthly_sales['forecast'] = monthly_sales['sales_amount'].rolling(3).mean()

In [111]:
monthly_sales

,year,month,sales_amount,forecast
0,2025,1,"285,044,244",NaN
1,2025,2,"232,519,409",NaN
2,2025,3,"277,334,956","264,966,203"
3,2025,4,"265,249,143","258,367,836"
4,2025,5,"287,306,794","276,630,298"
5,2025,6,"264,590,631","272,382,189"
6,2025,7,"251,976,292","267,957,906"
7,2025,8,"267,588,448","261,385,124"
8,2025,9,"270,588,605","263,384,448"
9,2025,10,"288,546,707","275,574,587"


### Product Level Monthly Forecast Demand

In [112]:
product_forecast = sales.groupby(['product_id', 'year', 'month'])['quantity_sold'].sum().reset_index()

In [114]:
product_forecast.head()

,product_id,year,month,quantity_sold
0,P00001,2025,4,22
1,P00003,2025,12,44
2,P00005,2025,9,45
3,P00005,2026,1,1
4,P00009,2025,4,11


In [115]:
product_forecast['forecast'] = (
    product_forecast
    .groupby('product_id')['quantity_sold']
    .transform(lambda x: x.rolling(3).mean())
)

In [121]:
product_forecast

,product_id,year,month,quantity_sold,forecast
0,P00001,2025,4,22,NaN
1,P00003,2025,12,44,NaN
2,P00005,2025,9,45,NaN
3,P00005,2026,1,1,NaN
4,P00009,2025,4,11,NaN
...,...,...,...,...,...
9574,P09994,2025,12,19,29
9575,P09997,2025,6,12,NaN
9576,P09998,2025,3,36,NaN
9577,P09998,2025,10,12,NaN


In [124]:
#checking if there are other numeric value in the column ['forecast']

product_forecast['forecast'].unique()

array([        nan, 16.66666667, 11.66666667, 59.66666667, 37.66666667,
       26.        , 48.33333333, 18.33333333, 33.33333333, 39.66666667,
       19.33333333, 24.66666667, 27.        , 28.33333333, 19.66666667,
       15.        , 21.66666667, 11.        , 29.        , 22.66666667,
       15.66666667, 34.33333333, 39.33333333, 23.66666667, 26.33333333,
       23.        , 23.33333333, 30.        , 34.66666667, 32.        ,
       29.33333333, 20.33333333, 31.33333333, 17.        , 22.33333333,
       18.66666667, 41.33333333, 25.        , 27.33333333, 14.66666667,
       14.33333333, 26.66666667, 35.        , 40.66666667, 39.        ,
       24.        , 24.33333333, 38.66666667, 29.66666667, 22.        ,
       17.33333333, 32.33333333, 37.        , 13.        , 41.66666667,
       30.33333333, 27.66666667, 35.66666667, 32.66666667, 37.33333333,
       25.66666667,  9.        , 18.        , 36.        , 21.33333333,
       36.33333333, 41.        , 34.        , 40.        , 13.66

In [140]:
# in powerbi dashboard to calculate the inventory value, which is Unit_price * current_stock
# I will merge product unit_price_ngn to inventory then write it as csv, then load it into powerbi

In [143]:
inventory.head(2)

,inventory_id,product_id,warehouse_id,current_stock,reorder_level,max_capacity,last_updated
0,I00001,P03994,W04507,1005,276,4027,2026-03-14
1,I00002,P01759,W09868,602,221,9745,2026-04-29


In [144]:
products.head(2)

,product_id,product_name,category,brand,unit_price_ngn,supplier_id
0,P00001,Peak Product 1,Food,Nestle,"6,273",S02287
1,P00002,Peak Product 2,Cleaning,Peak,"14,844",S00521


In [148]:
merge_inventory = inventory.merge(
    products[['product_id', 'unit_price_ngn']],
    on='product_id',
    how='left'
)

In [149]:
merge_inventory

,inventory_id,product_id,warehouse_id,current_stock,reorder_level,max_capacity,last_updated,unit_price_ngn
0,I00001,P03994,W04507,1005,276,4027,2026-03-14,"20,049"
1,I00002,P01759,W09868,602,221,9745,2026-04-29,"15,819"
2,I00003,P07241,W09951,1484,499,1095,2026-05-27,"20,576"
3,I00004,P00229,W09085,3559,306,8374,2026-04-03,"14,742"
4,I00005,P02681,W06466,2061,113,5267,2026-05-07,"14,105"
...,...,...,...,...,...,...,...,...
9995,I09996,P06454,W04643,891,242,4897,2026-04-23,"13,168"
9996,I09997,P05743,W02164,4974,196,1595,2026-05-31,"10,181"
9997,I09998,P05143,W05538,2211,188,8553,2026-05-09,"11,976"
9998,I09999,P02959,W00824,4268,92,1505,2026-04-03,"20,666"


In [151]:
merge_inventory['inventory_value'] = (
    merge_inventory['current_stock'] *
    merge_inventory['unit_price_ngn']
)

In [153]:
merge_inventory.to_csv("inventory_product.csv", index=False)

In [154]:
merge_inventory

,inventory_id,product_id,warehouse_id,current_stock,reorder_level,max_capacity,last_updated,unit_price_ngn,inventory_value
0,I00001,P03994,W04507,1005,276,4027,2026-03-14,"20,049","20,148,994"
1,I00002,P01759,W09868,602,221,9745,2026-04-29,"15,819","9,522,803"
2,I00003,P07241,W09951,1484,499,1095,2026-05-27,"20,576","30,535,007"
3,I00004,P00229,W09085,3559,306,8374,2026-04-03,"14,742","52,467,846"
4,I00005,P02681,W06466,2061,113,5267,2026-05-07,"14,105","29,069,642"
...,...,...,...,...,...,...,...,...,...
9995,I09996,P06454,W04643,891,242,4897,2026-04-23,"13,168","11,732,946"
9996,I09997,P05743,W02164,4974,196,1595,2026-05-31,"10,181","50,639,349"
9997,I09998,P05143,W05538,2211,188,8553,2026-05-09,"11,976","26,478,958"
9998,I09999,P02959,W00824,4268,92,1505,2026-04-03,"20,666","88,201,421"
